In [1]:
#Starts from here
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    # Identify columns with variance below the threshold
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [2]:
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_all_atomic_desc_RRCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_all_atomic_desc_RRCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(140, 250)
(35, 250)
(140, 347)
(35, 347)
(140, 637)
(35, 637)
(140, 13)
(35, 13)


In [3]:
merge_keys = ['ID', 'SMILES', 'Permeability']

merged_train = df_desc_train.merge(df_fp_train, on=merge_keys)
merged_train = merged_train.merge(df_emb_train, on=merge_keys)
merged_train = merged_train.merge(df_atomic_train, on=merge_keys)

merged_test = df_desc_test.merge(df_fp_test, on=merge_keys)
merged_test = merged_test.merge(df_emb_test, on=merge_keys)
merged_test = merged_test.merge(df_atomic_test, on=merge_keys)

In [4]:
X_train = merged_train.drop(columns=['ID', 'SMILES']).select_dtypes(include=['number'])
selected_final_features = features(X_train, target_column='Permeability')

train = pd.concat([merged_train[['ID', 'SMILES', 'Permeability']], X_train[selected_final_features]], axis=1)
test = merged_test[train.columns] 

print('selected_final_features', selected_final_features )
print("Final Train shape:", train.shape)
print("Final Test shape:", test.shape)

selected_final_features ['qed', 'SPS', 'MaxAbsPartialCharge', 'FpDensityMorgan1', 'BCUT2D_MRHI', 'AvgIpc', 'BalabanJ_x', 'Ipc', 'PEOE_VSA14', 'EState_VSA11', 'NumSaturatedRings', 'fr_alkyl_halide', 'fr_allylic_oxid', 'fr_morpholine', 'fr_unbrch_alkane', 'AdjacencyMatrix.6', 'AATS.12', 'AATS.67', 'AATS.80', 'AATS.93', 'AATS.95', 'AATS.96', 'ATSC.1', 'ATSC.6', 'ATSC.7', 'ATSC.8', 'ATSC.11', 'ATSC.16', 'ATSC.17', 'ATSC.20', 'ATSC.22', 'ATSC.24', 'ATSC.25', 'ATSC.26', 'ATSC.28', 'ATSC.32', 'ATSC.35', 'ATSC.44', 'ATSC.46', 'ATSC.64', 'ATSC.75', 'ATSC.80', 'ATSC.87', 'ATSC.95', 'ATSC.97', 'ATSC.98', 'ATSC.105', 'ATSC.106', 'AATSC.11', 'AATSC.12', 'AATSC.14', 'AATSC.15', 'AATSC.16', 'AATSC.34', 'AATSC.47', 'AATSC.49', 'AATSC.52', 'GATS.3', 'GATS.4', 'GATS.5', 'GATS.7', 'GATS.13', 'GATS.14', 'GATS.20', 'GATS.22', 'GATS.30', 'GATS.45', 'GATS.82', 'BCUT.3', 'Chi.15', 'AtomTypeEState.96', 'AtomTypeEState.129', 'AtomTypeEState.170', 'AtomTypeEState.173', 'AtomTypeEState.245', 'AtomTypeEState.249',

In [5]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [6]:
X_train = train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test[X_train.columns]
y_test = test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1214)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1214)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.022749 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 33481
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1005
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1483,0.3097,0.3851,0.6212,0.7885,0.7845,0.2291,0.3432,0.4786,0.5040,0.7118,0.6721
DecisionTreeRegressor,0.3475,0.4550,0.5895,0.1120,0.5706,0.5918,0.2947,0.3910,0.5429,0.3620,0.6369,0.5743
RandomForestRegressor,0.1470,0.3036,0.3834,0.6245,0.7911,0.7834,0.2402,0.3444,0.4901,0.4800,0.6940,0.6478
GradientBoostingRegressor,0.1592,0.3089,0.3989,0.5934,0.7710,0.7674,0.2200,0.3345,0.4691,0.5236,0.7252,0.6836
AdaBoostRegressor,0.1498,0.3048,0.3870,0.6173,0.7857,0.7846,0.2239,0.3375,0.4731,0.5153,0.7192,0.6535
XGBRegressor,0.1860,0.3272,0.4312,0.5248,0.7329,0.7250,0.2427,0.3557,0.4926,0.4746,0.6914,0.6535
ExtraTreesRegressor,0.1382,0.2891,0.3718,0.6469,0.8060,0.8083,0.2205,0.3339,0.4696,0.5226,0.7248,0.6797
LinearRegression,0.2615,0.4070,0.5114,0.3319,0.6752,0.6610,0.3341,0.3995,0.5780,0.2767,0.6180,0.5886
KNeighborsRegressor,0.2063,0.3384,0.4542,0.4728,0.7042,0.7116,0.3057,0.3740,0.5529,0.3382,0.6015,0.5306
SVR,0.1544,0.3025,0.3929,0.6056,0.7807,0.7902,0.2567,0.3750,0.5067,0.4442,0.6774,0.6038


In [7]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_features_rrck.csv')

In [8]:
X = train.drop(columns=['ID', 'SMILES', 'Permeability'])
y = train['Permeability']

rf = RandomForestRegressor(n_estimators=100, random_state=101, n_jobs=-1)
rf.fit(X, y)

importances = rf.feature_importances_
feature_names = X.columns


In [9]:
#Top 10 features
n = 10  
top_10_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_10_features = feature_names[top_10_indices].tolist() 

# Output the list
print("Top", 10, "features:\n")
print(top_10_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_10_features]], axis=1)
test_df = test[train.columns] 

Top 10 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440']


In [10]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 10)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 10)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071393 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 390
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 10
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-3.9167177575230765


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1568,0.3187,0.3960,0.5994,0.7774,0.7856,0.2369,0.3514,0.4867,0.4871,0.6987,0.6511
DecisionTreeRegressor,0.2872,0.4052,0.5359,0.2662,0.6421,0.6412,0.1942,0.3278,0.4407,0.5796,0.7617,0.7300
RandomForestRegressor,0.1534,0.3128,0.3916,0.6082,0.7807,0.7801,0.2402,0.3563,0.4901,0.4800,0.6939,0.6125
GradientBoostingRegressor,0.1663,0.3192,0.4078,0.5752,0.7639,0.7776,0.2239,0.3509,0.4732,0.5152,0.7178,0.6605
AdaBoostRegressor,0.1601,0.3131,0.4001,0.5910,0.7699,0.7692,0.2580,0.3836,0.5079,0.4414,0.6647,0.5976
XGBRegressor,0.1730,0.3220,0.4160,0.5579,0.7570,0.7638,0.2052,0.3281,0.4530,0.5557,0.7494,0.7136
ExtraTreesRegressor,0.1580,0.3147,0.3975,0.5963,0.7731,0.7785,0.2229,0.3432,0.4721,0.5174,0.7198,0.6674
LinearRegression,0.1591,0.3165,0.3989,0.5934,0.7712,0.7619,0.2623,0.3675,0.5122,0.4320,0.6733,0.5814
KNeighborsRegressor,0.1939,0.3522,0.4404,0.5045,0.7255,0.7168,0.2587,0.3496,0.5086,0.4399,0.6818,0.5916
SVR,0.1746,0.3238,0.4178,0.5540,0.7486,0.7651,0.2980,0.3875,0.5459,0.3548,0.6157,0.5543


In [11]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_10_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/_prediction_data_combined_top_10_features_rrck.csv')

In [12]:
#Top 20 features
n = 20  
top_20_indices = importances.argsort()[::-1][:n]  
top_20_features = feature_names[top_20_indices].tolist()  # convert to list

# Output the list
print("Top", 20, "features:\n")
print(top_20_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_20_features]], axis=1)
test_df = test[train.columns] 

Top 20 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL535', 'x_fine_emb_MFXL718', 'x_fine_emb_MFXL401', 'x_fine_emb_MFXL406', 'x_fine_emb_MFXL484', 'x_fine_emb_MFXL678', 'x_fine_emb_MFXL298', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL723', 'x_fine_emb_MFXL416']


In [13]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 20)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 20)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.065372 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 780
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 20
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-3.8650691917783675


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1457,0.3099,0.3817,0.6277,0.7931,0.7875,0.2383,0.3421,0.4881,0.4842,0.6994,0.6515
DecisionTreeRegressor,0.2104,0.3538,0.4587,0.4624,0.7424,0.7272,0.1968,0.3237,0.4436,0.5740,0.7589,0.7172
RandomForestRegressor,0.1405,0.3030,0.3748,0.6411,0.8008,0.7899,0.2387,0.3488,0.4885,0.4833,0.6972,0.6106
GradientBoostingRegressor,0.1661,0.3215,0.4075,0.5757,0.7634,0.7559,0.2317,0.3450,0.4813,0.4984,0.7081,0.6406
AdaBoostRegressor,0.1580,0.3144,0.3975,0.5962,0.7736,0.7742,0.2523,0.3736,0.5023,0.4537,0.6743,0.6003
XGBRegressor,0.1714,0.3284,0.4140,0.5621,0.7597,0.7602,0.2233,0.3488,0.4726,0.5165,0.7206,0.7149
ExtraTreesRegressor,0.1461,0.3083,0.3823,0.6267,0.7921,0.7843,0.2161,0.3369,0.4649,0.5320,0.7311,0.6909
LinearRegression,0.1674,0.3291,0.4092,0.5722,0.7596,0.7526,0.2404,0.3548,0.4903,0.4795,0.7068,0.6588
KNeighborsRegressor,0.1937,0.3557,0.4401,0.5050,0.7271,0.7094,0.2927,0.3597,0.5410,0.3664,0.6403,0.5784
SVR,0.1612,0.3147,0.4015,0.5881,0.7693,0.7768,0.2678,0.3675,0.5174,0.4203,0.6562,0.6227


In [14]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_20_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_top_20_features_rrck.csv')

In [15]:
#Top 50 features
n = 50  
top_50_indices = importances.argsort()[::-1][:n] 
top_50_features = feature_names[top_50_indices].tolist()  # convert to list

# Output the list
print("Top", 50, "features:\n")
print(top_50_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_50_features]], axis=1)
test_df = test[train.columns] 

Top 50 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL535', 'x_fine_emb_MFXL718', 'x_fine_emb_MFXL401', 'x_fine_emb_MFXL406', 'x_fine_emb_MFXL484', 'x_fine_emb_MFXL678', 'x_fine_emb_MFXL298', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL723', 'x_fine_emb_MFXL416', 'x_fine_emb_MFXL455', 'x_fine_emb_MFXL217', 'x_fine_emb_MFXL365', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL270', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL569', 'x_fine_emb_MFXL748', 'x_fine_emb_MFXL102', 'DPSA-3', 'x_fine_emb_MFXL339', 'GATS.4', 'x_fine_emb_MFXL285', 'x_fine_emb_MFXL719', 'x_fine_emb_MFXL370', 'x_fine_emb_MFXL464', 'x_fine_emb_MFXL510', 'maxsCH3', 'x_fine_emb_MFXL509', 'x_fine_emb_MFXL36', 'x_fine_emb_MFXL111', 'x_fine_emb_MFXL507', 'x_fine_emb_MFXL224', 'x_fine_emb_MFXL590', 'x_fine_emb_MFXL428', 'x_fine_emb_MFXL697', 'x_fin

In [16]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 50)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 50)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.088446 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1945
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 50
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


-1.65627716571962


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1348,0.2953,0.3671,0.6557,0.8100,0.8126,0.2346,0.3393,0.4844,0.4921,0.7048,0.6697
DecisionTreeRegressor,0.2581,0.3836,0.5080,0.3406,0.6639,0.6449,0.2155,0.3204,0.4643,0.5334,0.7338,0.6955
RandomForestRegressor,0.1318,0.2916,0.3630,0.6633,0.8148,0.8005,0.2417,0.3406,0.4917,0.4767,0.6941,0.6454
GradientBoostingRegressor,0.1344,0.2905,0.3666,0.6565,0.8106,0.8037,0.2230,0.3270,0.4722,0.5173,0.7212,0.6696
AdaBoostRegressor,0.1416,0.2971,0.3762,0.6383,0.7990,0.7886,0.2342,0.3507,0.4839,0.4929,0.7029,0.6420
XGBRegressor,0.1650,0.3198,0.4062,0.5785,0.7679,0.7540,0.2402,0.3491,0.4901,0.4800,0.6985,0.6606
ExtraTreesRegressor,0.1227,0.2801,0.3503,0.6865,0.8287,0.8159,0.2233,0.3302,0.4725,0.5166,0.7200,0.6751
LinearRegression,0.2691,0.4250,0.5187,0.3125,0.6642,0.6493,0.3012,0.3796,0.5488,0.3478,0.6416,0.5850
KNeighborsRegressor,0.1773,0.3325,0.4211,0.5470,0.7510,0.7430,0.2586,0.3474,0.5085,0.4402,0.6752,0.6353
SVR,0.1343,0.2931,0.3665,0.6569,0.8105,0.8163,0.2652,0.3671,0.5150,0.4258,0.6612,0.6111


In [17]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_50_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_top_50_features_rrck.csv')

In [18]:
#Top 100 features
n = 100  
top_100_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_100_features = feature_names[top_100_indices].tolist()  # convert to list

# Output the list
print("Top", 100, "features:\n")
print(top_100_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_100_features]], axis=1)
test_df = test[train.columns] 

Top 100 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL535', 'x_fine_emb_MFXL718', 'x_fine_emb_MFXL401', 'x_fine_emb_MFXL406', 'x_fine_emb_MFXL484', 'x_fine_emb_MFXL678', 'x_fine_emb_MFXL298', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL723', 'x_fine_emb_MFXL416', 'x_fine_emb_MFXL455', 'x_fine_emb_MFXL217', 'x_fine_emb_MFXL365', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL270', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL569', 'x_fine_emb_MFXL748', 'x_fine_emb_MFXL102', 'DPSA-3', 'x_fine_emb_MFXL339', 'GATS.4', 'x_fine_emb_MFXL285', 'x_fine_emb_MFXL719', 'x_fine_emb_MFXL370', 'x_fine_emb_MFXL464', 'x_fine_emb_MFXL510', 'maxsCH3', 'x_fine_emb_MFXL509', 'x_fine_emb_MFXL36', 'x_fine_emb_MFXL111', 'x_fine_emb_MFXL507', 'x_fine_emb_MFXL224', 'x_fine_emb_MFXL590', 'x_fine_emb_MFXL428', 'x_fine_emb_MFXL697', 'x_fi

In [19]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 100)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 100)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.163972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3887
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 100
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-1.280222867304622


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1502,0.3145,0.3876,0.6162,0.7855,0.7881,0.2217,0.3450,0.4708,0.5200,0.7242,0.6887
DecisionTreeRegressor,0.3242,0.4364,0.5694,0.1717,0.6075,0.5824,0.2515,0.3611,0.5015,0.4555,0.6793,0.6252
RandomForestRegressor,0.1323,0.2905,0.3637,0.6620,0.8141,0.8022,0.2342,0.3398,0.4839,0.4929,0.7047,0.6553
GradientBoostingRegressor,0.1443,0.2976,0.3799,0.6312,0.7960,0.7958,0.2140,0.3269,0.4626,0.5367,0.7327,0.6873
AdaBoostRegressor,0.1347,0.2959,0.3670,0.6558,0.8104,0.8001,0.2319,0.3484,0.4816,0.4979,0.7067,0.6525
XGBRegressor,0.1723,0.3238,0.4151,0.5598,0.7585,0.7528,0.2268,0.3389,0.4762,0.5090,0.7145,0.6654
ExtraTreesRegressor,0.1241,0.2771,0.3523,0.6829,0.8270,0.8148,0.2188,0.3296,0.4678,0.5262,0.7263,0.6618
LinearRegression,1.8661,1.0321,1.3661,-3.7679,0.2053,0.2067,1.2081,0.7510,1.0991,-1.6156,0.2803,0.4116
KNeighborsRegressor,0.1757,0.3210,0.4192,0.5510,0.7549,0.7646,0.2835,0.3716,0.5324,0.3863,0.6390,0.6113
SVR,0.1381,0.2911,0.3717,0.6471,0.8047,0.8134,0.2514,0.3664,0.5014,0.4557,0.6821,0.6445


In [20]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_100_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_top_100_features_rrck.csv')

In [21]:
#Top 200 features
n = 200  
top_200_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_200_features = feature_names[top_200_indices].tolist()  # convert to list

# Output the list
print("Top", 200, "features:\n")
print(top_200_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_200_features]], axis=1)
test_df = test[train.columns]

Top 200 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL535', 'x_fine_emb_MFXL718', 'x_fine_emb_MFXL401', 'x_fine_emb_MFXL406', 'x_fine_emb_MFXL484', 'x_fine_emb_MFXL678', 'x_fine_emb_MFXL298', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL723', 'x_fine_emb_MFXL416', 'x_fine_emb_MFXL455', 'x_fine_emb_MFXL217', 'x_fine_emb_MFXL365', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL270', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL569', 'x_fine_emb_MFXL748', 'x_fine_emb_MFXL102', 'DPSA-3', 'x_fine_emb_MFXL339', 'GATS.4', 'x_fine_emb_MFXL285', 'x_fine_emb_MFXL719', 'x_fine_emb_MFXL370', 'x_fine_emb_MFXL464', 'x_fine_emb_MFXL510', 'maxsCH3', 'x_fine_emb_MFXL509', 'x_fine_emb_MFXL36', 'x_fine_emb_MFXL111', 'x_fine_emb_MFXL507', 'x_fine_emb_MFXL224', 'x_fine_emb_MFXL590', 'x_fine_emb_MFXL428', 'x_fine_emb_MFXL697', 'x_fi

In [22]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 200)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 200)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.070258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7684
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 200
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1443,0.3025,0.3799,0.6313,0.7952,0.7974,0.2233,0.3392,0.4726,0.5165,0.7209,0.6751
DecisionTreeRegressor,0.3291,0.4314,0.5737,0.1591,0.5935,0.6155,0.2943,0.3994,0.5425,0.3629,0.6350,0.5553
RandomForestRegressor,0.1357,0.2886,0.3684,0.6532,0.8089,0.8018,0.2338,0.3371,0.4835,0.4939,0.7041,0.6511
GradientBoostingRegressor,0.1425,0.2868,0.3775,0.6359,0.7982,0.7942,0.2241,0.3472,0.4734,0.5147,0.7185,0.6702
AdaBoostRegressor,0.1382,0.2988,0.3718,0.6468,0.8049,0.7948,0.2241,0.3441,0.4734,0.5148,0.7187,0.6754
XGBRegressor,0.1679,0.3165,0.4097,0.5711,0.7624,0.7590,0.2360,0.3584,0.4858,0.4890,0.7024,0.6457
ExtraTreesRegressor,0.1253,0.2770,0.3539,0.6799,0.8249,0.8170,0.2162,0.3275,0.4649,0.5320,0.7300,0.6758
LinearRegression,0.6663,0.6331,0.8163,-0.7023,0.5021,0.5043,0.7618,0.5995,0.8728,-0.6493,0.4160,0.4574
KNeighborsRegressor,0.1784,0.3157,0.4224,0.5441,0.7537,0.7611,0.2889,0.3654,0.5375,0.3746,0.6321,0.5829
SVR,0.1372,0.2868,0.3704,0.6495,0.8077,0.8212,0.2326,0.3493,0.4823,0.4965,0.7063,0.6779


In [23]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_200_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_top_200_features_rrck.csv')

In [24]:
#Top 500 features
n = 500  
top_500_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_500_features = feature_names[top_500_indices].tolist()  # convert to list

# Output the list
print("Top", 500, "features:\n")
print(top_500_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_500_features]], axis=1)
test_df = test[train.columns]

Top 500 features:

['x_fine_emb_MFXL209', 'x_fine_emb_MFXL159', 'x_fine_emb_MFXL359', 'x_fine_emb_MFXL647', 'x_fine_emb_MFXL89', 'x_fine_emb_MFXL469', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL482', 'x_fine_emb_MFXL158', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL535', 'x_fine_emb_MFXL718', 'x_fine_emb_MFXL401', 'x_fine_emb_MFXL406', 'x_fine_emb_MFXL484', 'x_fine_emb_MFXL678', 'x_fine_emb_MFXL298', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL723', 'x_fine_emb_MFXL416', 'x_fine_emb_MFXL455', 'x_fine_emb_MFXL217', 'x_fine_emb_MFXL365', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL270', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL569', 'x_fine_emb_MFXL748', 'x_fine_emb_MFXL102', 'DPSA-3', 'x_fine_emb_MFXL339', 'GATS.4', 'x_fine_emb_MFXL285', 'x_fine_emb_MFXL719', 'x_fine_emb_MFXL370', 'x_fine_emb_MFXL464', 'x_fine_emb_MFXL510', 'maxsCH3', 'x_fine_emb_MFXL509', 'x_fine_emb_MFXL36', 'x_fine_emb_MFXL111', 'x_fine_emb_MFXL507', 'x_fine_emb_MFXL224', 'x_fine_emb_MFXL590', 'x_fine_emb_MFXL428', 'x_fine_emb_MFXL697', 'x_fi

In [25]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 500)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 500)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.079920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18998
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 500
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1548,0.3167,0.3935,0.6044,0.7786,0.7746,0.2333,0.3503,0.4830,0.4949,0.7053,0.6654
DecisionTreeRegressor,0.3773,0.4567,0.6143,0.0359,0.5394,0.5592,0.3268,0.4255,0.5716,0.2926,0.5957,0.5476
RandomForestRegressor,0.1455,0.3015,0.3815,0.6282,0.7934,0.7900,0.2373,0.3432,0.4871,0.4863,0.6986,0.6490
GradientBoostingRegressor,0.1544,0.2996,0.3930,0.6054,0.7793,0.7745,0.2149,0.3359,0.4636,0.5347,0.7325,0.6896
AdaBoostRegressor,0.1474,0.3013,0.3839,0.6235,0.7914,0.7931,0.2289,0.3494,0.4785,0.5044,0.7115,0.6667
XGBRegressor,0.1799,0.3184,0.4241,0.5405,0.7430,0.7377,0.2418,0.3565,0.4918,0.4764,0.6918,0.6504
ExtraTreesRegressor,0.1321,0.2879,0.3635,0.6624,0.8155,0.8102,0.2170,0.3344,0.4658,0.5302,0.7291,0.6690
LinearRegression,0.3038,0.4353,0.5512,0.2237,0.6537,0.6648,0.5839,0.5170,0.7641,-0.2641,0.4396,0.4571
KNeighborsRegressor,0.1817,0.3253,0.4263,0.5357,0.7429,0.7577,0.2990,0.3764,0.5468,0.3527,0.6175,0.5612
SVR,0.1442,0.2970,0.3798,0.6315,0.7967,0.8101,0.2544,0.3666,0.5044,0.4493,0.6767,0.6329


In [26]:
result_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/combined_top_500_features_rrck.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/combined_features/prediction_data_combined_top_500_features_rrck.csv')